# Bollinger Band Reversal (BBR)

## Import Libs

In [3]:
import pandas as pd 
import numpy as np
import os
from pandas import DataFrame, Series
import plotly.graph_objects as go

## Functions

### Get FE Data

In [1]:
def get_fe_price_data(
        filename: str = "../output/FE_V2_GBPUSD_15mins_1yr_End_20250311.csv"
        ) -> DataFrame:
    """
    Return the FE price data as a DatetimeIndexed 
    DataFrame set to US/Eastern TZ
    """
    PATH = os.getcwd()
    df = pd.read_csv(filename)
    df["Date"] = pd.DatetimeIndex(df["Date"], tz="US/Eastern")
    df.set_index("Date", inplace=True)
    return df

### Simulate Short Positions (Range-Based)

In [188]:
def range_short_gains(
        df: Series, 
        high: Series, 
        low: Series,
        close: Series,
        signal_name: str,
        sl_pct_range: int,
        tp_pct_range: int,
        range_type: str = "ADR"
        ):
    """Get the pip gain and apply to df"""

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None
    
    if df[signal_name] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = high.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        # get stop loss time:
        sl_window = high.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out

        for i in range(len(sl_window)): 
            if sl_window.iloc[i] >= (df["Close"] + df[range_type] * sl_pct_range ):
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = sl_window.iloc[i:i+1].index[0] 
                break
        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD
        # Take profit price
        tp = df["Close"] - (df[range_type] * tp_pct_range)
        
        # trade window
        tp_window = low[START+TD:sl_ts+TD]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = df["Close"] - tp
        sl_pips = df["Close"] - (df["Close"] + df[range_type] * sl_pct_range) 
        if tp_window.min() <= tp:
            win = 1
            gain = df["Close"] - tp
        else:
            loss = 1
            if stop is True:
                gain = sl_pips
            else:
                gain = df["Close"] - close_window.iloc[-1] if close_window.empty is False else 0
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end
    
    return data

# Set new columns 
# gains_cols = [ 
#     "Win", "Loss", 
#     "TP", "SL", 
#     "Gain",
#     "Trade_Start", "Trade_End"
#     ]

#df[pct_50_idr_cols] = df.apply(get_bear_bbr_pip_gain, axis=1, args=[df["High"], df["Low"], df["Close"]], result_type='expand')


### Simulate Long Positions (Range-Based)

In [189]:
def range_long_gains(
        df: Series, 
        high: Series, 
        low: Series,
        close: Series,
        signal_name: str,
        sl_pct_range: int,
        tp_pct_range: int,
        range_type: str = "ADR"
        ):
    """Get the pip gain and apply to df"""

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None
    
    if df[signal_name] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = low.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        # get stop loss time:
        sl_window = low.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out

        for i in range(len(sl_window)): 
            if sl_window.iloc[i] <= (df["Close"] - df[range_type] * sl_pct_range ):
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = sl_window.iloc[i:i+1].index[0] 
                break
        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD

        # Take profit price
        tp = df["Close"] + (df[range_type] * tp_pct_range)
           
        # trade window
        tp_window = high[START+TD:sl_ts+TD]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = tp - df["Close"]
        sl_pips = (df["Close"] - df[range_type] * sl_pct_range) - df["Close"] 
        if tp_window.max() >= tp:
            win = 1
            gain = tp - df["Close"]
        else:
            loss = 1
            if stop is True:
                gain = sl_pips
            else:
                gain = close_window.iloc[-1] - df["Close"] if close_window.empty is False else 0
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end, \
        
    return data

# Set new columns 
# long_gains_cols = [
#     "Win", "Loss", 
#     "TP", "SL", 
#     "Gain",
#     "Trade_Start", "Trade_End",
#     ]

#df[pct_50_idr_cols] = df.apply(get_bull_bbr_pip_gain, axis=1, args=[df["High"], df["Low"], df["Close"]], result_type='expand')


### Simulate Long/Short Trend Position

In [132]:
def trend_gains(
        df: Series, 
        close: Series,
        sma: Series,
        signal_name: str,
        trade_direction: int
        ):
    """Get the pip gain and apply to df
    
    - trade_direction must be:
        - Long: 1
        - Short -1
    """

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None
    
    if df[signal_name] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = close.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        sma_window = sma.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out
        
        # find exit timestamp:
        for i in range(len(close_window)): 
            # exit condition
            if trade_direction == 1:
                exit_condition = close_window.iloc[i] < sma_window.iloc[i]
            elif trade_direction == -1:
                exit_condition = close_window.iloc[i] > sma_window.iloc[i]
            # get stop loss time:
            if exit_condition:
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = close_window.iloc[i:i+1].index[0] 
                break
        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD if (START+TD) < END else START

        # Win / Loss / Gain / Pips
        if trade_direction == 1:
            gain = close[sl_ts] - df["Close"]
        elif trade_direction == -1:
            gain = df["Close"] - close[sl_ts]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = df["ATR4"]
        sl_pips = -(df["ATR4"])
        win = 1 if gain > 0 else 0
        loss = 1 if gain < 0 else 0
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end, \
        
    return data



In [64]:
# Set new columns 
gains_cols = [ 
    "Win", "Loss", 
    "TP", "SL", 
    "Gain",
    "Trade_Start", "Trade_End"
    ]

### Trade Stats

In [133]:
def trade_stats(df: DataFrame, signal_name: str):
    win_count = df.query(f"{signal_name} == True and Win > 0")[f"{signal_name}"].count()
    loss_count = df.query(f"{signal_name} == True and Loss > 0")[f"{signal_name}"].count()
    total_trades = win_count + loss_count
    win_rate = win_count/total_trades * 100
    win = df.query(f"{signal_name} == True and Gain > 0")["Gain"]
    loss = df.query(f"{signal_name} == True and Gain < 0")["Gain"]
    win_avg_pips = win.mean()
    loss_avg_pips = loss.mean()
    win_pips = win.sum()
    loss_pips = loss.sum()
    total_pips = win_pips + loss_pips

    stats = {
        "Win_Count": win_count,
        "Loss_Count": loss_count,
        "Total_Trades": total_trades,
        "Win_Rate": win_rate,
        "Avg_Win": win_avg_pips,
        "Avg_Loss": loss_avg_pips,
        "Win_Pips": win_pips,
        "Loss_Pips": loss_pips,
        "Total_Pips": total_pips
    }

    return stats

## Breakout

In [178]:
pd.options.display.max_rows = 100
data_end_date: str = "20250311"
path: str = f"./price_data/FE_GBPUSD_15mins_1yr_End_{data_end_date}.csv"
bull_bo_df = get_fe_price_data(filename=path)
bear_bo_df = get_fe_price_data(filename=path)

long_signal = "BBU_BO"
short_signal = "BBL_BO"

# long
bull_bo_df[gains_cols] = bull_bo_df.apply(
    trend_gains,
    axis=1,
    args=[bull_bo_df["Close"], bull_bo_df["SMA4"], long_signal, 1],
    result_type='expand'
)
# short
bear_bo_df[gains_cols] = bear_bo_df.apply(
    trend_gains,
    axis=1,
    args=[bear_bo_df["Close"], bear_bo_df["SMA4"], short_signal, -1],
    result_type='expand'
)

bull_bo_gains_df = trade_stats(bull_bo_df, long_signal)
bear_bo_gains_df = trade_stats(bear_bo_df, short_signal)

bo_gains_df = pd.DataFrame(data=[bull_bo_gains_df, bear_bo_gains_df],
             index=[long_signal,short_signal]
             )

bo_gains_df

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/850889442.py:9: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/850889442.py:9: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)


,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
BBU_BO,167,315,482,34.647303,0.001191,-0.000723,0.198855,-0.227665,-0.028810
BBL_BO,170,291,461,36.876356,0.001229,-0.000664,0.208915,-0.193270,0.015645


In [184]:
bull_bo_df[[*gains_cols, "Close_Pct_SMA", "RSI", "Body", "ATR"]].query("Loss > 0").nsmallest(10,"Gain")

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,Close_Pct_SMA,RSI,Body,ATR
Date,,,,,,,,,,,
2024-12-06 08:30:00-05:00,0.0,1.0,0.001490,-0.001490,-0.003720,2024-12-06 08:45:00-05:00,2024-12-06 09:00:00-05:00,0.296050,79.877277,0.003285,0.000986
2024-09-06 08:30:00-04:00,0.0,1.0,0.002937,-0.002937,-0.003580,2024-09-06 08:45:00-04:00,2024-09-06 09:00:00-04:00,0.377979,75.426498,0.004205,0.001419
2025-01-20 09:30:00-05:00,0.0,1.0,0.003265,-0.003265,-0.003335,2025-01-20 09:45:00-05:00,2025-01-20 10:15:00-05:00,0.772954,76.543566,0.002545,0.002510
2024-10-30 10:00:00-04:00,0.0,1.0,0.002432,-0.002432,-0.003290,2024-10-30 10:15:00-04:00,2024-10-30 10:45:00-04:00,0.426167,66.467809,0.003230,0.001717
2024-07-11 08:30:00-04:00,0.0,1.0,0.002254,-0.002254,-0.002860,2024-07-11 08:45:00-04:00,2024-07-11 09:15:00-04:00,0.491804,87.051831,0.006815,0.000980
2024-07-16 08:15:00-04:00,0.0,1.0,0.000585,-0.000585,-0.002735,2024-07-16 08:30:00-04:00,2024-07-16 08:30:00-04:00,0.095223,66.879079,0.000755,0.000544
2025-01-17 10:00:00-05:00,0.0,1.0,0.002352,-0.002352,-0.002450,2025-01-17 10:15:00-05:00,2025-01-17 10:30:00-05:00,0.245332,60.903841,0.002750,0.001604
2024-04-23 04:30:00-04:00,0.0,1.0,0.001234,-0.001234,-0.002420,2024-04-23 04:45:00-04:00,2024-04-23 05:00:00-04:00,0.274646,71.234479,0.002155,0.001101
2024-09-20 03:00:00-04:00,0.0,1.0,0.001315,-0.001315,-0.002165,2024-09-20 03:15:00-04:00,2024-09-20 03:30:00-04:00,0.268756,82.067400,0.001440,0.000950


## SMA Breakout

In [186]:
pd.options.display.max_rows = 100
data_end_date: str = "20250311"
path: str = f"./price_data/FE_GBPUSD_15mins_1yr_End_{data_end_date}.csv"
bull_sma_bo_df = get_fe_price_data(filename=path)
bear_sma_bo_df = get_fe_price_data(filename=path)

long_signal = "Bull_SMA_BO"
short_signal = "Bear_SMA_BO"

# long
bull_sma_bo_df[gains_cols] = bull_sma_bo_df.apply(
    trend_gains,
    axis=1,
    args=[bull_sma_bo_df["Close"], bull_sma_bo_df["SMA4"], long_signal, 1],
    result_type='expand'
)
# short
bear_sma_bo_df[gains_cols] = bear_sma_bo_df.apply(
    trend_gains,
    axis=1,
    args=[bear_sma_bo_df["Close"], bear_sma_bo_df["SMA4"], short_signal, -1],
    result_type='expand'
)

bull_sma_bo_gains_df = trade_stats(bull_sma_bo_df, long_signal)
bear_sma_bo_gains_df = trade_stats(bear_sma_bo_df, short_signal)

sma_bo_gains_df = pd.DataFrame(data=[bull_sma_bo_gains_df, bear_sma_bo_gains_df],
             index=[long_signal,short_signal]
             )

sma_bo_gains_df

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/850889442.py:9: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/850889442.py:9: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)


,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
Bull_SMA_BO,62,72,134,46.268657,0.001410,-0.000623,0.087405,-0.044855,0.04255
Bear_SMA_BO,57,95,152,37.500000,0.001299,-0.000639,0.074015,-0.060665,0.01335


## Bollinger Band Reversals

In [209]:
pd.options.display.max_rows = 100
data_end_date: str = "20250311"
path: str = f"./price_data/FE_GBPUSD_15mins_1yr_End_{data_end_date}.csv"
bull_bbr_df = get_fe_price_data(filename=path)
bear_bbr_df = get_fe_price_data(filename=path)

long_signal = "Bull_BBR_V2"
short_signal = "Bear_BBR_V2"

# long
bull_bbr_df[gains_cols] = bull_bbr_df.apply(
    range_long_gains,
    axis=1,
    args=[bull_bbr_df["High"],bull_bbr_df["Low"],bull_bbr_df["Close"], 
          long_signal, 1.5, 2.5],
    result_type='expand',
    range_type="ATR4"
)
# short
bear_bbr_df[gains_cols] = bear_bbr_df.apply(
    range_short_gains,
    axis=1,
    args=[bear_bbr_df["High"],bear_bbr_df["Low"],bear_bbr_df["Close"], 
          short_signal, 1.5, 2.5],
    result_type='expand',
    range_type="ATR4"
)

bull_bbr_gains_df = trade_stats(bull_bbr_df, long_signal)
bear_bbr_gains_df = trade_stats(bear_bbr_df, short_signal)

bbr_gains_df = pd.DataFrame(data=[bull_bbr_gains_df, bear_bbr_gains_df],
             index=[long_signal,short_signal]
             )

bbr_gains_df

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/850889442.py:9: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/850889442.py:9: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)


,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
Bull_BBR_V2,183,309,492,37.195122,0.002349,-0.001330,0.544886,-0.344341,0.200544
Bear_BBR_V2,150,354,504,29.761905,0.002351,-0.001313,0.458520,-0.404528,0.053992


In [205]:
bull_bbr_df[[*gains_cols, "Bull_BBR_C1","Bull_BBR_C2","Bull_BBR_C3","Bull_BBR_C4"]].query("Win > 0 or Loss >0").iloc[0:100]

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,Bull_BBR_C1,Bull_BBR_C2,Bull_BBR_C3,Bull_BBR_C4
Date,,,,,,,,,,,
2024-03-13 06:00:00-04:00,1.0,0.0,0.001950,-0.001170,0.001950,2024-03-13 06:15:00-04:00,2024-03-13 16:45:00-04:00,True,NaN,NaN,NaN
2024-03-13 23:00:00-04:00,0.0,1.0,0.000956,-0.000574,-0.000574,2024-03-13 23:15:00-04:00,2024-03-13 23:30:00-04:00,True,NaN,NaN,NaN
2024-03-14 08:30:00-04:00,0.0,1.0,0.003409,-0.002046,-0.002046,2024-03-14 08:45:00-04:00,2024-03-14 09:00:00-04:00,True,NaN,NaN,NaN
2024-03-14 09:30:00-04:00,0.0,1.0,0.002947,-0.001768,-0.001768,2024-03-14 09:45:00-04:00,2024-03-14 10:15:00-04:00,NaN,NaN,True,NaN
2024-03-17 19:30:00-04:00,0.0,1.0,0.000647,-0.000388,-0.000388,2024-03-17 19:45:00-04:00,2024-03-17 20:00:00-04:00,True,NaN,NaN,NaN
2024-03-17 20:15:00-04:00,1.0,0.0,0.000850,-0.000510,0.000850,2024-03-17 20:30:00-04:00,2024-03-18 10:30:00-04:00,True,NaN,NaN,NaN
2024-03-18 11:30:00-04:00,0.0,1.0,0.001778,-0.001067,0.000095,2024-03-18 11:45:00-04:00,2024-03-18 16:45:00-04:00,NaN,True,NaN,NaN
2024-03-18 20:00:00-04:00,0.0,1.0,0.000694,-0.000416,-0.000416,2024-03-18 20:15:00-04:00,2024-03-18 20:30:00-04:00,NaN,True,NaN,NaN
2024-03-19 02:15:00-04:00,0.0,1.0,0.001419,-0.000851,-0.000851,2024-03-19 02:30:00-04:00,2024-03-19 03:00:00-04:00,True,NaN,NaN,NaN
